# NASA FIRMS — download dos CSVs anuais

Este notebook consulta o manifesto público FIRMS e baixa os arquivos anuais por país para MODIS, VIIRS S-NPP (identificador `viirs-snpp`, exibido como **Data** no portal) e VIIRS NOAA-20. Não requer `MAP_KEY` nem login.

O volume observado na elaboração deste notebook é de aproximadamente 5,85 GB; manifesto, anos disponíveis e volume são consultados dinamicamente. O processamento dos CSVs baixados é feito no notebook `02_tratamento_firms.ipynb`.

Uso de IA: ferramentas de IA auxiliaram na elaboração e revisão deste notebook, conforme a declaração do projeto no README.

## 1. Dependências
Execute a instalação apenas se as bibliotecas não estiverem disponíveis.

In [ ]:
# %pip install requests pandas tqdm

In [ ]:
from pathlib import Path
import csv
import re
import tempfile
from urllib.parse import quote

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm

## 2. Configuração
O modo de simulação é ativado inicialmente. Altere `DRY_RUN` para `False` somente quando quiser iniciar os downloads.

In [ ]:
BASE_URL = 'https://firms.modaps.eosdis.nasa.gov'
MANIFEST_URL = f'{BASE_URL}/data/country/yearly_summary_files.txt'
# Funciona quando o kernel inicia na raiz do projeto ou dentro de dataset_extractor.
CWD = Path.cwd().resolve()
DATASET_DIR = CWD if CWD.name == 'dataset_extractor' else CWD / 'dataset_extractor'
OUTPUT_DIR = DATASET_DIR / 'csv' / 'raw'
REPORT_PATH = DATASET_DIR / 'csv' / 'download_report.csv'
SOURCES = {'modis': 'MODIS', 'viirs-snpp': 'VIIRS S-NPP', 'viirs-jpss1': 'VIIRS NOAA-20'}
COUNTRIES = ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Guyana', 'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela', 'French_Guiana']
DRY_RUN = True
CHUNK_SIZE = 1024 * 1024
TIMEOUT = (15, 120)
MAX_RETRIES = 5
print(f'DRY_RUN={DRY_RUN}; diretório de dados: {OUTPUT_DIR.resolve()}')

In [ ]:
retry = Retry(total=MAX_RETRIES, connect=MAX_RETRIES, read=MAX_RETRIES, status=MAX_RETRIES, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504], allowed_methods=['GET'], raise_on_status=False)
adapter = HTTPAdapter(max_retries=retry)
session = requests.Session()
session.headers.update({'User-Agent': 'NASA-FIRMS-South-America-Notebook/1.0'})
session.mount('https://', adapter)
session.mount('http://', adapter)

## 3. Consultar o manifesto e preparar a seleção
São selecionados somente CSVs de país listados no manifesto; arquivos ZIP, README e outros países são ignorados.

In [ ]:
response = session.get(MANIFEST_URL, timeout=TIMEOUT)
response.raise_for_status()
manifest_rows = list(csv.DictReader(response.text.splitlines()))
filename_pattern = re.compile(r'^(modis|viirs-snpp|viirs-jpss1)_(\d{4})_(.+)\.csv$')
selected = []
for item in manifest_rows:
    match = filename_pattern.fullmatch((item.get('filename') or '').strip())
    if not match:
        continue
    source, year_text, country = match.groups()
    if source not in SOURCES or country not in COUNTRIES:
        continue
    item_size = int(item['size'])
    filename = item['filename'].strip()
    url = f"{BASE_URL}/data/country/{source}/{year_text}/{quote(filename)}"
    local_path = OUTPUT_DIR / source / year_text / filename
    selected.append({'source': source, 'year': int(year_text), 'country': country, 'filename': filename, 'size': item_size, 'updated': item.get('updated', ''), 'url': url, 'path': local_path})

if not selected:
    raise RuntimeError('O manifesto não retornou arquivos para as fontes e países configurados.')
print(f'Arquivos selecionados: {len(selected):,}')
print(f'Tamanho total: {sum(x["size"] for x in selected) / 1e9:.3f} GB | {sum(x["size"] for x in selected) / 1024**3:.3f} GiB')

In [ ]:
import pandas as pd
preview = pd.DataFrame([{k: x[k] for k in ['source', 'year', 'country', 'size']} for x in selected])
display(preview.groupby('source').agg(arquivos=('size', 'size'), bytes=('size', 'sum'), primeiro_ano=('year', 'min'), ultimo_ano=('year', 'max')))
display(preview.groupby('country').agg(arquivos=('size', 'size'), bytes=('size', 'sum')).sort_values('arquivos', ascending=False))
available_years = {source: sorted(preview.loc[preview.source == source, 'year'].unique()) for source in SOURCES}
missing = [(source, year, country) for source, years in available_years.items() for year in years for country in COUNTRIES if not ((preview.source == source) & (preview.year == year) & (preview.country == country)).any()]
print(f'Combinações país/ano sem arquivo no manifesto: {len(missing)}')
if missing:
    display(pd.DataFrame(missing, columns=['source', 'year', 'country']).head(50))

## 4. Baixar arquivos
Downloads são gravados como `.part`, verificados pelo tamanho e só então movidos ao nome `.csv`. Arquivos válidos já existentes são ignorados. Falhas individuais são registradas e não interrompem a fila.

In [ ]:
report_rows = []
for item in tqdm(selected, desc='Arquivos FIRMS'):
    path = item['path']
    status, error, local_size = 'dry_run', '', path.stat().st_size if path.exists() else 0
    if not DRY_RUN:
        part_path = path.with_name(path.name + '.part')
        try:
            if path.exists() and path.stat().st_size == item['size']:
                status, local_size = 'skipped', path.stat().st_size
            else:
                path.parent.mkdir(parents=True, exist_ok=True)
                part_path.unlink(missing_ok=True)
                with session.get(item['url'], stream=True, timeout=TIMEOUT) as download:
                    download.raise_for_status()
                    with part_path.open('wb') as output:
                        for block in download.iter_content(chunk_size=CHUNK_SIZE):
                            if block:
                                output.write(block)
                local_size = part_path.stat().st_size
                if local_size != item['size']:
                    status = 'size_mismatch'
                    raise ValueError(f"Tamanho recebido {local_size}; esperado {item['size']} bytes")
                part_path.replace(path)
                status = 'downloaded'
        except Exception as exc:
            error = f'{type(exc).__name__}: {exc}'
            if status != 'size_mismatch':
                status = 'failed'
            part_path.unlink(missing_ok=True)
            local_size = path.stat().st_size if path.exists() else 0
    report_rows.append({'source': item['source'], 'year': item['year'], 'country': item['country'], 'filename': item['filename'], 'url': item['url'], 'expected_size': item['size'], 'local_size': local_size, 'status': status, 'error': error})

REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(report_rows, columns=['source', 'year', 'country', 'filename', 'url', 'expected_size', 'local_size', 'status', 'error']).to_csv(REPORT_PATH, index=False)
display(pd.DataFrame(report_rows).groupby('status').size().rename('arquivos'))
print(f'Relatório salvo em {REPORT_PATH.resolve()}')

In [ ]:
if not DRY_RUN:
    failures = [r for r in report_rows if r['status'] in {'failed', 'size_mismatch'}]
    print(f'Arquivos com falha: {len(failures)}')
    if failures:
        display(pd.DataFrame(failures)[['filename', 'status', 'error']])
else:
    print('Simulação concluída; nenhum CSV foi baixado. Para iniciar, altere DRY_RUN para False e execute novamente desde a configuração.')